# Machine Learning on text with TF-IDF and LSA

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy
import pandas
import pathlib
import seaborn
import sklearn.decomposition
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline

## Getting data
Execute the cell below to retrieve the two corpora of this lab.

The first one is a collection of movie plots from Wikipedia, with metadata such as the release year, the title or the genre.

The second one, 20 Newsgroups, gathers the messages of 20 discussion forums: it will be used for the latent semantic analysis.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wikipedia-movie-plots
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
movie_plot_path = "wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Loading the data

Execute the cell below. It loads the CSV file into a pandas dataframe, `df`, and displays its columns and its size: you should find 34,886 movies. A row is a movie; the `Genre` column gives its genre, and `Plot` the summary of its plot, in English.

In [ ]:
df = pandas.read_csv(movie_plot_path)
print(f"Columns: {', '.join(df.columns)}")
print(f"Dataframe size: {df.shape}")

## Selecting data

To keep genres with enough movies, we only keep the 5 most present ones. The most frequent genre, `unknown`, is obviously not one of them.

What are the genres that we keep? Build a new dataframe, `clean_df`, that holds only the movies of those genres, and plot the number of movies per genre. How many movies are left?

**Useful documentation**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.drop.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)
- [`seaborn.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html)

In [ ]:
# Your code here

### Additional hints

`df["Genre"].value_counts()` counts the movies of each genre, from the most frequent to the least; `.drop("unknown")` removes `unknown` from this count, and `.index.values[:5]` gives the first 5 genres left. `df["Genre"].isin(genres)` is `True` for the movies of one of these genres: it selects the rows of `df`. `seaborn.countplot(clean_df, x="Genre")` plots the number of movies of each genre.

You should keep `drama`, `comedy`, `horror`, `action` and `thriller`, that is 13,574 movies.

```python
counts = df["Genre"].value_counts().drop(...)
genres = counts.index.values[:...]
print(f"Selected genres: {', '.join(genres)}")
print(f"Number of selected movies: {counts[genres].sum()}")

clean_df = df[...]
seaborn.countplot(clean_df, x="Genre")
```

### Solution

In [ ]:
counts = df["Genre"].value_counts().drop("unknown")
genres = counts.index.values[:5]
print(f"Selected genres: {', '.join(genres)}")
print(f"Number of selected movies: {counts[genres].sum()}")

In [ ]:
clean_df = df[df["Genre"].isin(genres)]
seaborn.countplot(clean_df, x="Genre")

## TF-IDF representation

Convert the plots `x` into vectors with `TfidfVectorizer`: remove the stop words and keep only the 3000 most frequent words. Call the vectorizer `vectorizer` and the resulting matrix `X`, then display the size of `X` and the first 10 words of the vocabulary.

**Useful documentation**

- [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
x = clean_df["Plot"]

# Your code here

### Additional hints

`stop_words="english"` removes the English stop words, the plots' language, and `max_features=3000` keeps the 3000 most frequent words. `fit_transform(x)` learns the vocabulary and the idf weights, then returns the matrix: one row per movie, one column per word. `get_feature_names_out()` gives the columns' words, in alphabetical order.

You should get a matrix of size `(13574, 3000)`.

```python
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=...
)
X = vectorizer.fit_transform(...)
feature_names = vectorizer.get_feature_names_out()
print(f"Matrix size: {X.shape}")
print(f"First 10 columns: {', '.join(feature_names[:10])}")
```

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000
)
X = vectorizer.fit_transform(x)
feature_names = vectorizer.get_feature_names_out()
print(f"Matrix size: {X.shape}")
print(f"First 10 columns: {', '.join(feature_names[:10])}")

## TF-IDF weights of a movie

Display the plot of the first movie (`x.iloc[0]`) and the 5 words with the highest weight in its row of `X`, with their weight: do they say what the movie is about?

Then display, for `time` and for `clouseau` (the inspector of *The Pink Panther*), the number of plots that contain the word, its $df_i$, and its idf, which `vectorizer.idf_` gives. Find what the slide "TF-IDF — Why $df_i$?" says: which of the two words weighs the most when it appears, and why?

Finally, compare these idf to the $\ln\frac{N}{df_i}$ of the course's formula, where $N$ is the number of plots. `TfidfVectorizer` computes a smoothed idf, $\ln\frac{1 + N}{1 + df_i} + 1$, then divides each row of `X` by its norm: what difference does that make?

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Your code here

### Additional hints

`X[0]` is the row of the first movie, a sparse matrix: `X[0].toarray()[0]` turns it into an array of 3000 weights, one per word of `feature_names`. `numpy.argsort(weights_0)` sorts its indices from the lowest weight to the highest: `[::-1][:5]` gives those of the 5 highest.

`vectorizer.vocabulary_` maps each word to its column, and `vectorizer.idf_[column]` gives its idf. `(X[:, column] > 0).sum()` counts the plots that contain the word, and `X.shape[0]` is the number of plots; `numpy.log` is the natural logarithm.

You should find `throwing`, `quiet` and `violent` on top, and idf values of 2.16 for `time` and 8.12 for `clouseau`.

```python
weights_0 = X[0].toarray()[0]
print(f"Plot of the first movie: {x.iloc[0]}")
for index in numpy.argsort(weights_0)[::-1][:5]:
  print(f"{feature_names[index]}: {weights_0[index]:.2f}")

n_movies = X.shape[0]
for word in ["time", "clouseau"]:
  column = vectorizer.vocabulary_[word]
  df_i = ...
  course_idf = ...
  print(
    f"{word}: in {df_i} plots, idf {vectorizer.idf_[column]:.2f}, "
    f"ln(N / df) {course_idf:.2f}"
  )
```

### Solution

In [ ]:
weights_0 = X[0].toarray()[0]
print(f"Plot of the first movie: {x.iloc[0]}")
for index in numpy.argsort(weights_0)[::-1][:5]:
  print(f"{feature_names[index]}: {weights_0[index]:.2f}")

n_movies = X.shape[0]
for word in ["time", "clouseau"]:
  column = vectorizer.vocabulary_[word]
  # df_i: the number of plots that contain the word
  df_i = (X[:, column] > 0).sum()
  course_idf = numpy.log(n_movies / df_i)
  print(
    f"{word}: in {df_i} plots, idf {vectorizer.idf_[column]:.2f}, "
    f"ln(N / df) {course_idf:.2f}"
  )

# throwing, quiet and violent tell the movie's story: fairly rare words (each in
# 250 to 400 plots), hence with a high idf, that appear in this short plot.
# time is the most widespread word of the vocabulary (4260 plots out of 13574): it
# has the lowest idf, 2.16. clouseau, in 10 plots, has the highest, 8.12.
# The course's formula gives 1.16 and 7.21: smoothing adds about 1 to each idf, so
# that a word present in every plot still weighs 1 instead of 0.
# The words stay ranked in the same order: the ideas are the same, the values
# change, and normalizing the rows changes them again.

## A corpus for LSA

Latent semantic analysis (LSA) is a truncated SVD of the documents × words matrix: each axis groups words that appear together. For it to show anything, the corpus needs well-marked themes. Movie plots don't have them: their axes fill up with first names (`joe`, `jack`, `mary`). So we switch to 20 Newsgroups, whose messages come from 20 thematic forums: sports, computing, religion, politics…

Execute the cell below. It loads the 11,314 messages of the corpus's training part, and removes their header (sender, subject…), which stops at the first blank line.

In [ ]:
# Each folder is a forum, then one message per file
messages = [
  p.read_text(encoding="latin-1").partition("\n\n")[2]
  for p in sorted(pathlib.Path("20news-bydate-train").glob("*/*"))
]

print(f"{len(messages)} messages retrieved")
print(messages[0][:300])

## TF-IDF representation of the messages

As for the movies, convert `messages` to vectors with `TfidfVectorizer`: remove the stop words and keep the 3000 most frequent words. Also remove the words present in more than 30% of the messages, or in fewer than 5 messages.

Display the size of the matrix and the average number of non-zero values per row: how many of the 3000 words does a message use?

**Useful documentation**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Your code here

### Additional hints

`stop_words` and `max_features` work as they did for the movies. `max_df=0.3` removes the words present in more than 30% of the documents (a float is a share of the documents) and `min_df=5` those present in fewer than 5 documents (an integer is a number of documents).

`X_messages.shape` gives the size of the matrix and `X_messages.nnz` its number of non-zero values. You should get `(11314, 3000)` and about 55 words per message.

```python
message_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=..., max_df=..., min_df=...
)
X_messages = message_vectorizer.fit_transform(...)
feature_names_messages = message_vectorizer.get_feature_names_out()
print(f"Matrix size: {X_messages.shape}")
print(f"Words per message: {...:.0f}")
```

### Solution

In [ ]:
message_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000, max_df=0.3, min_df=5
)
X_messages = message_vectorizer.fit_transform(messages)
feature_names_messages = message_vectorizer.get_feature_names_out()
print(f"Matrix size: {X_messages.shape}")
print(f"Words per message: {X_messages.nnz / X_messages.shape[0]:.0f}")

## Latent Semantic Analysis

Use `TruncatedSVD` to compute an LSA with 6 axes on `X_messages`.

Display the singular value of each axis, the share of variance it keeps, and the share the 6 axes keep together. What share of the variance do they keep? Does the axis with the largest singular value keep the largest variance?

**Useful documentation**

- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
# Your code here

### Additional hints

`n_components=6` sets the number of axes, `n_iter=100` refines the computation and `random_state=42` gives the same axes from one run to the next. After `svd.fit(X_messages)`, the attributes `singular_values_` and `explained_variance_ratio_` hold one value per axis.

You should get a singular value of 16 for axis 1, and about 2.8% of the variance for the 6 axes.

```python
svd = sklearn.decomposition.TruncatedSVD(...)
svd.fit(...)

print(f"Singular values: {svd.singular_values_.round(2)}")
print(f"Share of variance per axis: {...}")
print(f"Share of variance of the 6 axes: {...:.1%}")
```

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=6, n_iter=100, random_state=42)
svd.fit(X_messages)

print(f"Singular values: {svd.singular_values_.round(2)}")
print(f"Share of variance per axis: {svd.explained_variance_ratio_.round(4)}")
print(f"Share of variance of the 6 axes: {svd.explained_variance_ratio_.sum():.1%}")

## Important words

The 6 axes keep less than 3% of the variance: that's normal, and it isn't a failure. A message uses only about 55 of the 3000 words, and most of the variance comes from these choices, specific to each message: it would take about 500 axes to keep half of it. LSA retains only what many messages share: judge an axis by its words, not by its share of variance.

Axis 1 isn't a theme. `TruncatedSVD` doesn't center the matrix, none of whose values is negative: this axis has only positive coefficients, and it gives the corpus's average vocabulary. That's how it has the largest singular value without keeping the largest variance. Themes are read from axis 2 on.

The sign of a coefficient carries the meaning: an axis opposes the pole of its positive coefficients to the pole of its negative ones. For each axis, display the 8 most positive words and the 8 most negative words. Then, for axes 2 to 6, name each pole: what theme is it about?

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Your code here

### Additional hints

`svd.components_` has one row per axis and one column per word. `numpy.argsort(svd.components_, axis=1)` sorts each row: for an axis, `indices` goes from the most negative coefficient to the most positive one. So `indices[:8]` gives the 8 most negative ones, `indices[::-1][:8]` the 8 most positive ones, and `feature_names_messages[indices[:8]]` the corresponding words.

You should find `god` and `jesus` at one end of axis 2, `windows` and `card` at the other. Only the opposition matters: which pole is the positive one depends only on the computation.

```python
n_words = 8

for number, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positive = ...
  negative = ...
  print(f"Axis {number}")
  print(f"  + {', '.join(positive)}")
  print(f"  - {', '.join(negative)}")
```

### Solution

In [ ]:
n_words = 8

# argsort sorts each axis's coefficients, from the most negative to the most positive
for number, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positive = feature_names_messages[indices[::-1][:n_words]]
  negative = feature_names_messages[indices[:n_words]]
  print(f"Axis {number}")
  print(f"  + {', '.join(positive)}")
  print(f"  - {', '.join(negative)}")

# Axis 1: the average vocabulary. Its coefficients are all positive: the words shown
# as negative are just the closest to 0.
# Axis 2: religion against computing
# Axis 3: religion against sports
# Axis 4: cryptography against religion and sports
# Axis 5: hardware against software
# Axis 6: cryptography against the Middle East
# The poles aren't pure (`windows` on the religion side of axis 3): the axes have to
# stay orthogonal, so a theme spreads over several axes.

## Supervised Approach

Back to the movies: we'll try to predict a movie's genre from its plot.

To do so, start by extracting the genre of each movie into a pandas series `y`. scikit-learn's classifiers take these strings as they are.

`drama` has 6 times as many movies as `thriller`. What accuracy does a model that always answers `drama` get? That's the score to beat.

**Useful documentation**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Your code here

### Additional hints

`clean_df["Genre"]` gives the series of genres. `y.value_counts(normalize=True)` gives the share of each genre instead of its number of movies: that of `drama` is the accuracy of a model that always answers `drama`.

You should find 43.9%.

```python
y = clean_df[...]

shares = y.value_counts(normalize=...)
display(shares)

print(f"Accuracy when always answering drama: {shares[...]:.1%}")
```

### Solution

In [ ]:
y = clean_df["Genre"]

shares = y.value_counts(normalize=True)
display(shares)

print(f"Accuracy when always answering drama: {shares['drama']:.1%}")

## Training

Train a logistic regression that takes the TF-IDF vectors of the plots as input and predicts the genre. It must do better than the constant model of the previous section.

Evaluate it with a 5-fold cross-validation, with `cross_val_predict`. The vocabulary and the idf weights are part of the model: so that they're only learned on each fold's training data, put the `TfidfVectorizer` and the classifier in a single pipeline (`make_pipeline`), applied to the texts `x`. The file is sorted by the movies' origin: so that the folds don't follow these blocks, shuffle them with `StratifiedKFold`.

Display the accuracy, the macro-F1 and the confusion matrix. The macro-F1 is the mean of the F1 of each genre: `thriller` counts as much as `drama` in it, whereas the accuracy depends mostly on the large genres. Write this evaluation in a function that receives the model to put after the vectorizer: it will be used three more times.

Then compare:

- the same logistic regression with `class_weight="balanced"`, which gives more weight to the movies of the small genres: what does the macro-F1 gain, what does the accuracy lose, which genres change in the confusion matrix?
- an LSA between the vectorizer and the logistic regression: insert `TruncatedSVD` in the pipeline, with 5 then 100 axes. What do we lose compared to the 3000 words?

**Useful documentation**

- [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [`sklearn.metrics.confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
# Your code here

### Additional hints

`make_pipeline(vectorizer, model)` chains the steps. A pipeline is itself a model: `make_pipeline(TruncatedSVD(...), LogisticRegression(...))` is passed to the function as `model`, to insert the LSA. `LogisticRegression(max_iter=200)` gives it enough iterations to converge, and `TruncatedSVD(5, random_state=0)` keeps 5 axes.

`StratifiedKFold(5, shuffle=True, random_state=0)` shuffles the movies and keeps the share of each genre in each fold; `cross_val_predict(model, x, y, cv=folds)` returns, for each movie, the prediction of a model trained without it. `f1_score(y, predictions, average="macro")` gives the macro-F1. `confusion_matrix(y, predictions, normalize="true")` gives, for each true genre (a row), the share of its movies predicted in each genre (a column), genres in alphabetical order (`sorted(genres)`); `seaborn.heatmap` displays it.

You should get an accuracy of 0.66 and a macro-F1 of 0.55 with the logistic regression alone.

```python
def evaluate_pipeline(model):
  model = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    model,
  )
  folds = sklearn.model_selection.StratifiedKFold(...)
  predictions = sklearn.model_selection.cross_val_predict(...)

  accuracy = sklearn.metrics.accuracy_score(y, predictions)
  macro_f1 = sklearn.metrics.f1_score(...)
  print(f"Accuracy: {accuracy:.2f}, macro-F1: {macro_f1:.2f}")

  confusion_matrix = sklearn.metrics.confusion_matrix(...)
  seaborn.heatmap(
    confusion_matrix,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap="Blues",
  )
  plt.xlabel("Predicted genre")
  plt.ylabel("True genre")
  plt.show()


evaluate_pipeline(sklearn.linear_model.LogisticRegression(max_iter=200))
evaluate_pipeline(
  sklearn.linear_model.LogisticRegression(class_weight=..., max_iter=200)
)
evaluate_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(..., random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)
```

### Solution

In [ ]:
def evaluate_pipeline(model):
  # The vectorizer is part of the pipeline: on each cross-validation fold, it
  # only learns its vocabulary and idf weights from the fold's training data
  model = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    model,
  )
  # The file is sorted by origin: shuffle so that the folds don't follow these
  # blocks
  folds = sklearn.model_selection.StratifiedKFold(5, shuffle=True, random_state=0)
  predictions = sklearn.model_selection.cross_val_predict(model, x, y, cv=folds)

  # Accuracy is the share of correct predictions: the large genres weigh the most
  # in it
  accuracy = sklearn.metrics.accuracy_score(y, predictions)
  # Macro-F1 is the unweighted mean of each genre's F1: the small genres count as
  # much as the large ones
  macro_f1 = sklearn.metrics.f1_score(y, predictions, average="macro")
  print(f"Accuracy: {accuracy:.2f}, macro-F1: {macro_f1:.2f}")

  # One row per true genre, one column per predicted genre: each cell gives the
  # share of the true genre's movies predicted in that genre
  confusion_matrix = sklearn.metrics.confusion_matrix(y, predictions, normalize="true")
  seaborn.heatmap(
    confusion_matrix,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap="Blues",
  )
  plt.xlabel("Predicted genre")
  plt.ylabel("True genre")
  plt.show()

In [ ]:
# Logistic regression alone
evaluate_pipeline(sklearn.linear_model.LogisticRegression(max_iter=200))

# Accuracy 0.66: better than the constant model's 0.44. But macro-F1 0.55: the
# small genres are poorly recognized, above all thriller, of which only 11% of the
# movies are predicted thriller

In [ ]:
# Rebalancing: each genre weighs as much in the training
evaluate_pipeline(
  sklearn.linear_model.LogisticRegression(class_weight="balanced", max_iter=200)
)

# Accuracy 0.63 (-0.03), macro-F1 0.58 (+0.03): the model predicts the small
# genres more often. It finds 42% of the thrillers instead of 11%, and 66% of the
# action movies instead of 38%, at the expense of the dramas (60% instead of 80%).
# Which one to keep depends on the use: the accuracy if only the number of movies
# correctly classified matters, the macro-F1 if every genre matters

In [ ]:
# LSA with 5 axes between the vectorizer and the logistic regression
evaluate_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(5, random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)

# Accuracy 0.46, macro-F1 0.23: barely better than the constant model, and
# thriller is never predicted anymore. 5 axes only keep the vocabulary that many
# plots share, not what sets the genres apart

In [ ]:
# LSA with 100 axes
evaluate_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(100, random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)

# Accuracy 0.60, macro-F1 0.49: better, but still below the 3000 words. LSA keeps
# the words that appear together in many plots, and loses the rarer words that
# carry part of the genre. It is used to explore a corpus and to shrink its
# representation, not here to classify better

## Going further

TF-IDF and LSA remain good baselines, fast and easy to interpret. To find the themes of a corpus, today (2026) one rather starts from sentence embeddings, grouped by proximity: that's what [BERTopic](https://maartengr.github.io/BERTopic/) does, then describing each group by its most characteristic words, with a variant of TF-IDF.